In [1]:
import pandas as pd
import altair as alt
from pathlib import Path

In [2]:
data_path = Path("../data/eic2025_genero.xlsx")

df = pd.read_excel(
    data_path,
    sheet_name="01",
    header=None
)

df.head(12)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,INEGI. Encuesta Intercensal 2025. Tabulados co...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Índice
1,Fecha de elaboración: 22/09/2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Estimadores de indicadores demográficos con en...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Género 1
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Entidad federativa,Estimador,Población total en viviendas particulares habi...,NaN,NaN,Relación mujeres-hombres1,Edad mediana2,NaN,NaN,NaN,Índice de envejecimiento3,NaN,NaN,NaN
6,NaN,NaN,Total,Mujeres,Hombres,NaN,Total,Mujeres,Hombres,Diferencia mujeres-hombres,Total,Mujeres,Hombres,Diferencia mujeres-hombres
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Estados Unidos Mexicanos,Valor,130393389,67678902,62714487,107.915898,31.886397,32.84969,30.905819,1.943871,68.217369,75.601924,61.021698,14.580226
9,Estados Unidos Mexicanos,Error estándar,498654.437209,258672.85178,243014.051027,0.09141,0.032303,0.033454,0.032617,0.046723,0.279674,0.336972,0.253929,0.421936


In [3]:
display(df.iloc[5:15, :10])

,0,1,2,3,4,5,6,7,8,9
5,Entidad federativa,Estimador,Población total en viviendas particulares habi...,NaN,NaN,Relación mujeres-hombres1,Edad mediana2,NaN,NaN,NaN
6,NaN,NaN,Total,Mujeres,Hombres,NaN,Total,Mujeres,Hombres,Diferencia mujeres-hombres
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Estados Unidos Mexicanos,Valor,130393389,67678902,62714487,107.915898,31.886397,32.84969,30.905819,1.943871
9,Estados Unidos Mexicanos,Error estándar,498654.437209,258672.85178,243014.051027,0.09141,0.032303,0.033454,0.032617,0.046723
10,Estados Unidos Mexicanos,Límite inferior de confianza,129573174.339672,67253422.450582,62314763.920358,107.765542,31.833263,32.794663,30.852168,1.867018
11,Estados Unidos Mexicanos,Límite superior de confianza,131213603.660328,68104381.549418,63114210.079642,108.066255,31.93953,32.904716,30.95947,2.020723
12,Estados Unidos Mexicanos,Coeficiente de variación,0.382423,0.382206,0.387493,0.084705,0.101306,0.101839,0.105538,2.403614
13,01 Aguascalientes,Valor,1534416,798001,736415,108.362948,29.76264,30.888426,28.757369,2.131057
14,01 Aguascalientes,Error estándar,58302.677495,30294.876541,28452.452847,1.022401,0.377157,0.425279,0.437382,0.610054


In [5]:
columns = {
    0: "state",
    1: "estimator",
    2: "population_total",
    3: "population_women",
    4: "population_men",
    5: "sex_ratio",
    6: "median_age_total",
    7: "median_age_women",
    8: "median_age_men",
    9: "median_age_gap",
    10: "aging_index_total",
    11: "aging_index_women",
    12: "aging_index_men",
    13: "aging_index_gap"
}

In [6]:
demographics = (
    # The file has also bonds, whe need just the value
    df.loc[df[1] == "Valor", list(columns)]
    .rename(columns=columns)
    .reset_index(drop=True)
)

numeric_cols = demographics.columns.drop(["state", "estimator"])

demographics[numeric_cols] = demographics[numeric_cols].apply(
    pd.to_numeric, errors="coerce"
)

demographics.head()

,state,estimator,population_total,population_women,population_men,sex_ratio,median_age_total,median_age_women,median_age_men,median_age_gap,aging_index_total,aging_index_women,aging_index_men,aging_index_gap
0,Estados Unidos Mexicanos,Valor,130393389,67678902,62714487,107.915898,31.886397,32.849690,30.905819,1.943871,68.217369,75.601924,61.021698,14.580226
1,01 Aguascalientes,Valor,1534416,798001,736415,108.362948,29.762640,30.888426,28.757369,2.131057,55.170495,60.968010,49.467775,11.500235
2,02 Baja California,Valor,3978593,2008265,1970328,101.925415,31.407915,31.921056,30.922530,0.998526,60.640175,66.351290,55.044319,11.306971
3,03 Baja California Sur,Valor,955465,476192,479273,99.357151,31.365006,31.671014,31.070821,0.600193,56.989274,59.891941,54.163772,5.728169
4,04 Campeche,Valor,967098,497706,469392,106.032058,31.820339,32.537724,31.064330,1.473394,63.536100,68.012866,59.230818,8.782048


In [7]:
#Cleaning the names of the states
demographics["state"] = (
    demographics["state"]
    .str.replace(r"^\d{2}\s+", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [8]:
mexico = demographics.iloc[0]

aging_data = pd.DataFrame({
    "sex": ["Women", "Men"],
    "median_age": [
        mexico["median_age_women"],
        mexico["median_age_men"]
    ],
    "aging_index": [
        mexico["aging_index_women"],
        mexico["aging_index_men"]
    ]
})

aging_data

,sex,median_age,aging_index
0,Women,32.849690,75.601924
1,Men,30.905819,61.021698


In [9]:
median_age = aging_data[["sex", "median_age"]]

chart = (
    alt.Chart(median_age)
    .mark_circle(size=250)
    .encode(
        x=alt.X(
            "median_age:Q",
            title="Median age (years)",
            scale=alt.Scale(domain=[28, 35])
        ),
        y=alt.Y(
            "sex:N",
            title=None,
            sort=["Women", "Men"]
        ),
        color=alt.Color(
            "sex:N",
            scale=alt.Scale(
                domain=["Women", "Men"],
                range=["#7952A3", "#258C95"]
            ),
            legend=None
        ),
        tooltip=[
            "sex:N",
            alt.Tooltip("median_age:Q", format=".1f")
        ]
    )
    .properties(
        width=550,
        height=120,
        title="Women in Mexico are older than men"
    )
)

chart

alt.Chart(...)

In [10]:
aging_states = demographics.iloc[1:][
    ["state", "aging_index_women", "aging_index_men"]
].copy()

aging_long = aging_states.melt(
    id_vars="state",
    value_vars=["aging_index_women", "aging_index_men"],
    var_name="sex",
    value_name="aging_index"
)

aging_long["sex"] = aging_long["sex"].map({
    "aging_index_women": "Women",
    "aging_index_men": "Men"
})

In [11]:
state_order = (
    aging_states
    .sort_values("aging_index_women", ascending=False)["state"]
    .tolist()
)

base = alt.Chart(aging_long).encode(
    y=alt.Y(
        "state:N",
        sort=state_order,
        title=None,
        axis=alt.Axis(labelLimit=180)
    )
)

lines = base.mark_line(
    color="#C9C5CF",
    strokeWidth=2
).encode(
    x=alt.X(
        "aging_index:Q",
        title="Aging index"
    ),
    detail="state:N"
)

points = base.mark_circle(size=85).encode(
    x="aging_index:Q",
    color=alt.Color(
        "sex:N",
        scale=alt.Scale(
            domain=["Women", "Men"],
            range=["#7952A3", "#258C95"]
        ),
        title=None
    ),
    tooltip=[
        "state:N",
        "sex:N",
        alt.Tooltip("aging_index:Q", format=".1f")
    ]
)

aging_chart = (
    (lines + points)
    .properties(
        width=650,
        height=650,
        title=alt.TitleParams(
            text="Women have a higher aging index across Mexican states",
            subtitle="Aging index by sex and state, Mexico, 2025",
            anchor="start"
        )
    )
    .configure_view(stroke=None)
    .configure_axis(
        grid=False,
        labelFontSize=11,
        titleFontSize=12
    )
    .configure_title(
        fontSize=19,
        color="#292633"
    )
)

aging_chart

alt.LayerChart(...)